
# Agentic Personal Research Assistant — Colab Step-by-Step (Updated)
Welcome! This notebook teaches while you build. By the end, you'll have an Agentic AI Research Assistant that:
- Searches arXiv for papers (MVP; you can add Semantic Scholar later)
- Stores results in a Chroma vector database (memory)
- Runs a LangGraph workflow (Planner -> Retriever -> Summarizer -> Reviewer)
- Produces a concise, citation-aware literature summary

Tip: run cells in order. For each step: read "Why?" and "What this cell does", then run it and check "Success signals".



## 0) Runtime Check (Colab vs Local)
Why?
- Confirms you are on Colab (paths/permissions).
- Shows Python version and GPU availability (relevant for local LLMs).

What this cell does
- Detects Colab import, prints platform info, and checks for GPU via nvidia-smi.

Success signals
- You see Running in Google Colab: True (if applicable).
- GPU presence is optional; API-based LLMs work without it.


In [ ]:

import sys, os, platform, subprocess

def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

print("Running in Google Colab:", in_colab())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# GPU check (useful for local Llama)
try:
    out = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT).decode()
    print("GPU detected")
except Exception:
    print("No GPU (OK for API providers)")



## 1) Install Dependencies
Why?
Pull in the building blocks:
- langchain: tool use, retrieval, chains
- langgraph: graph-based orchestration for multi-agent flows
- chromadb: vector memory (persistent)
- faiss-cpu: fast similarity search (optional internal use)
- arxiv: query arXiv programmatically
- tiktoken: token utilities
- openai / google-generativeai: API LLM providers (optional)
- transformers / bitsandbytes: for local Llama (optional)

Success signals
- No fatal pip errors. Warnings are fine.


In [ ]:

!pip -q install langchain langgraph chromadb faiss-cpu arxiv tiktoken openai google-generativeai langchain-openai
# Optional for local Llama 3.1 (4-bit)
!pip -q install transformers accelerate bitsandbytes huggingface_hub



## 2) Configure Provider Flags & API Keys (Optional)
Why?
- Lets you switch summarizer providers without rewriting code.

What this cell does
- Sets a PROVIDER flag: "gemini" | "openai" | "local" | "extractive"
- Reads API keys from environment variables.

Success signals
- It prints which keys are set (True/False). Both False is fine (you will use extractive fallback).


In [ ]:

import os

# Choose summarizer backend: "gemini", "openai", "local", or "extractive"
PROVIDER = "gemini"

# Optional: set keys (leave empty to skip)
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY","")
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY","")

HAS_GEMINI = bool(os.environ.get("GEMINI_API_KEY"))
HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("Gemini key set:", HAS_GEMINI, "| OpenAI key set:", HAS_OPENAI, "| Provider:", PROVIDER)



## 3) Initialize LLM + Embeddings
Why?
- Embeddings enable semantic memory (Chroma).
- LLM yields fluent summaries (fallback is extractive if no API/local model).

What this cell does
- If OPENAI_API_KEY is present, configures ChatOpenAI and OpenAIEmbeddings.
- Otherwise, it continues without embeddings (memory recall disabled, but core flow still works).

Success signals
- Prints whether LLM & embeddings are configured.


In [ ]:

llm = None
embedding_model = None

if HAS_OPENAI:
    try:
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
        embedding_model = OpenAIEmbeddings()
    except Exception as e:
        print("OpenAI init failed:", e)

print("LLM configured:", bool(llm))
print("Embeddings configured:", bool(embedding_model))



## 4) Create Vector Memory (ChromaDB)
Why?
- Persists knowledge across runs.
- Enables semantic recall.

What this cell does
- Creates a Chroma collection and small helper functions.

Success signals
- Prints the Chroma path. No exceptions.


In [ ]:

from langchain.vectorstores import Chroma
from langchain.docstore.document import Document
from typing import List

CHROMA_DIR = "/content/chroma_research_memory"
vectorstore = Chroma(
    collection_name="research_memory",
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR
)

def add_to_memory(docs: List[Document]):
    if docs:
        vectorstore.add_documents(docs)
        vectorstore.persist()

def recall_from_memory(query: str, k: int = 5):
    if embedding_model is None:
        return []
    try:
        return vectorstore.similarity_search(query, k=k)
    except Exception as e:
        print("Recall failed (likely no embeddings backend).", e)
        return []

print("Chroma ready at", CHROMA_DIR)



## 5) Paper Retrieval Utility — arXiv (MVP)
Why?
- No API key required; stable abstracts and PDF links, perfect for demos.

What this cell does
- Queries arXiv and returns a list of dicts per paper.

Success signals
- A test call like search_arxiv("agentic AI") returns non-empty results.


In [ ]:

import arxiv, time
from typing import Dict, Any, List

def search_arxiv(query: str, max_results: int = 5) -> List[Dict[str, Any]]:
    client = arxiv.Client()
    results = client.results(arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    ))
    items = []
    for r in results:
        items.append({
            "source": "arxiv",
            "title": r.title,
            "authors": [a.name for a in r.authors],
            "summary": r.summary,
            "published": r.published.isoformat() if r.published else "",
            "pdf_url": r.pdf_url,
            "link": r.entry_id
        })
        time.sleep(3)  # polite throttle
    return items



## 6) Convert Results -> Documents
Why?
- Vector stores and chains use Document objects (text + metadata).

What this cell does
- Converts retrieval output into standardized Documents.

Success signals
- A sample conversion prints a Document object.


In [ ]:

def to_documents(results: List[Dict[str, Any]]) -> List[Document]:
    docs = []
    for r in results:
        text = r.get("summary") or r.get("title","")
        meta = {k:v for k,v in r.items() if k != "summary"}
        docs.append(Document(page_content=text, metadata=meta))
    return docs

# Smoke test
example_docs = to_documents([{"source":"test","title":"Hello","summary":"World"}])
print("Doc example:", example_docs[0])



## 7) Summarization Strategy (Provider-agnostic)
Why?
- One function supports Gemini, OpenAI, Local Llama, or Extractive (no LLM).
- Keeps your code clean and easy to switch providers.

What this cell does
- Builds stable bullets and prompts a summary with inline [n] citations.


In [ ]:

def format_bullets(docs: List[Document], max_items=6):
    out = []
    for i, d in enumerate(docs[:max_items], 1):
        title = d.metadata.get("title","(untitled)")
        link = d.metadata.get("pdf_url") or d.metadata.get("link") or ""
        snippet = d.page_content.strip().split("\n")[0][:220]
        bullet = f"[{i}] {title} — {snippet}"
        if link:
            bullet += f" [link]({link})"
        out.append(bullet)
    return out


In [ ]:

def summarize_findings(query: str, docs: List[Document]) -> str:
    import os
    bullets = format_bullets(docs)
    if not bullets:
        return "(no results)"
    if PROVIDER == "gemini" and bool(os.environ.get("GEMINI_API_KEY")):
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        model = genai.GenerativeModel("gemini-2.5-flash")
        prompt = f"Topic: {query}\nWrite a 140-180 word literature-review paragraph with 2-4 inline [#] citations referring to the bullets below:\n" + "\n".join(bullets)
        return model.generate_content(prompt).text
    if PROVIDER == "openai" and bool(os.environ.get("OPENAI_API_KEY")):
        from langchain_openai import ChatOpenAI
        llm_local = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
        prompt = f"You are a scholarly assistant. Topic: {query}\nWrite a 140-180 word literature-review paragraph with 2-4 inline [#] citations referring to the bullets below:\n" + "\n".join(bullets)
        return llm_local.invoke(prompt).content
    if PROVIDER == "local":
        # Lazy-load local Llama only if selected
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
        # If Llama weights require token, set: os.environ["HUGGING_FACE_HUB_TOKEN"] = "hf_xxx"
        model_id = "meta-llama/Llama-3.1-8B-Instruct"
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        tok = AutoTokenizer.from_pretrained(model_id, token=os.environ.get("HUGGING_FACE_HUB_TOKEN",""))
        mdl = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,
                                                   torch_dtype=torch.bfloat16, device_map="auto",
                                                   token=os.environ.get("HUGGING_FACE_HUB_TOKEN",""))
        pipe = pipeline("text-generation", model=mdl, tokenizer=tok, device_map="auto")
        prompt = f"You are a scholarly assistant. Topic: {query}\nWrite a 140-180 word literature-review paragraph with 2-4 inline [#] citations referring to the bullets below:\n" + "\n".join(bullets)
        out = pipe(prompt, max_new_tokens=192, do_sample=False, temperature=0.2,
                   top_p=0.9, repetition_penalty=1.1,
                   pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)[0]["generated_text"]
        return out[len(prompt):].strip() if out.startswith(prompt) else out.strip()
    # Extractive fallback
    return "\n".join("- "+b for b in bullets)



## 8) Build the Agent Flow with LangGraph
Why?
- A clear graph separates concerns and eases debugging/extension.

What this cell does
- Defines nodes: planner -> retriever -> summarizer -> reviewer, then compiles the graph.


In [ ]:

from langgraph.graph import StateGraph, START, END

class State(dict): pass

def planner(state: State) -> State:
    # arXiv-only MVP plan
    state["plan"] = {"subtasks": [{"action":"arxiv","k":5}, {"action":"recall","k":4}]}
    print("Plan:", state["plan"])
    return state

def retriever(state: State) -> State:
    q = state.get("query","")
    collected = []
    for t in state["plan"]["subtasks"]:
        if t["action"] == "arxiv":
            collected += search_arxiv(q, max_results=t["k"])
        elif t["action"] == "recall":
            for d in recall_from_memory(q, k=t["k"]):
                collected.append({"source":"memory","title":d.metadata.get("title","(memory)"),
                                  "summary":d.page_content, **{k:v for k,v in d.metadata.items() if k!='title'}})
    docs = to_documents(collected)
    add_to_memory(docs)
    state["docs"] = docs
    print("Retrieved", len(docs), "docs (including memory).")
    return state

def summarizer_node(state: State) -> State:
    q = state.get("query","")
    docs = state.get("docs",[])
    state["summary"] = summarize_findings(q, docs)
    return state

def reviewer(state: State) -> State:
    s = state.get("summary","")
    state["review"] = "ok" if (len(s) > 100 and ("[1]" in s or "[2]" in s)) else "needs more content/citations"
    return state

graph = StateGraph(State)
graph.add_node("planner", planner)
graph.add_node("retriever", retriever)
graph.add_node("summarizer", summarizer_node)
graph.add_node("reviewer", reviewer)
graph.add_edge(START, "planner")
graph.add_edge("planner", "retriever")
graph.add_edge("retriever", "summarizer")
graph.add_edge("summarizer", "reviewer")
graph.add_edge("reviewer", END)
app = graph.compile()
print("Graph compiled.")



## 9) Run an Example Query
Why?
- Smoke test end-to-end.

Success signals
- Review: ok (or similar).
- A paragraph summary (LLM) or bullet list (extractive).


In [ ]:

topic = "autonomous multi-agent AI systems for literature review"
result = app.invoke(State({"query": topic}))
print("Review:", result.get("review"))
print("\n--- Summary ---\n")
print(result.get("summary","(no summary)"))



## 10) Inspect Memory
Why?
- Validate that items were persisted and are recallable.

Success signals
- You see items and links on related queries.


In [ ]:

hits = recall_from_memory("agentic AI literature review", k=3) or []
for i, d in enumerate(hits, 1):
    meta = d.metadata
    link = meta.get("pdf_url") or meta.get("link") or ""
    print(f"[{i}] {meta.get('title','(no title)')} — {meta.get('source','')}")
    print(link, "\n")



## 11) Exercises (Learn by Doing)
1) Modify the plan: add a year filter before storing docs.  
2) Reviewer++: Require >= 2 [n] markers and >= 3 source links.  
3) Gradio UI: Build a 3-tab interface (Sources / Summary / Memory).  
4) Full-text RAG (advanced): download OA PDFs, chunk, and ground summaries via retrieval over chunks.
